# Orinoco Digital — Modelo DEMO (Fase 5)

## ⚠️ Lee esto antes de ejecutar nada

**Este notebook NO predice donde perforar. No es asesoria de inversion ni de ingenieria.**

Entrena un clasificador sobre **datos sinteticos**. Lo unico real son las
distribuciones de cada variable, tomadas de la tabla 1 del
[USGS Fact Sheet 2009-3028](https://pubs.usgs.gov/fs/2009/3028/).

**La limitacion que no se puede maquillar:** la etiqueta objetivo la calcula
`scripts/generar-features-demo.mjs` con una formula que nos inventamos. El
modelo no aprende geologia — aprende a recuperar nuestra propia formula.

Por eso las metricas que veras abajo **no significan nada sobre el mundo real**.
Un AUC alto aqui solo dice que el bosque aleatorio sabe imitar una suma
ponderada, cosa que ya sabiamos.

**Para que sirve entonces:** para ensenar el flujo completo —rejilla, features,
entrenamiento, mapa de calor sobre el terreno— con datos que no fingimos que
sean reales. Es material didactico, no un producto.

Ver `MODEL_CARD.md` en el repositorio.

---

**Por que Colab y no local:** la laptop de desarrollo tiene 8 GB de RAM y no
tiene Python instalado. Entrenar aqui es la decision correcta, no un parche.

In [ ]:
import json

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

print('Listo. scikit-learn y pandas cargados.')

## 1. Cargar el dataset

Genera primero el CSV en tu maquina:

```bash
node scripts/generar-features-demo.mjs
```

Eso crea `data/raw/features-demo.csv`. Subelo con la celda siguiente.

In [ ]:
from google.colab import files

subidos = files.upload()  # elige features-demo.csv
datos = pd.read_csv('features-demo.csv')

print(f'{len(datos)} celdas')
print(datos['etiqueta_sintetica'].value_counts(normalize=True).round(3))
datos.head()

## 2. Entrenar

Las coordenadas **no** se usan como features. Si se incluyeran, el modelo
memorizaria posiciones en vez de aprender la relacion entre variables, y el
mapa resultante seria un calco del dataset.

In [ ]:
COLUMNAS = [
    'porosidad_pct',
    'saturacion_agua_pct',
    'espesor_arena_neta_ft',
    'profundidad_m',
    'gravedad_api',
]

X = datos[COLUMNAS]
y = datos['etiqueta_sintetica']

X_ent, X_prueba, y_ent, y_prueba = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

modelo = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=5, random_state=42
)
modelo.fit(X_ent, y_ent)

prob_prueba = modelo.predict_proba(X_prueba)[:, 1]
print(classification_report(y_prueba, modelo.predict(X_prueba)))
print(f'AUC: {roc_auc_score(y_prueba, prob_prueba):.3f}')

print('\nImportancia de cada variable:')
for col, imp in sorted(
    zip(COLUMNAS, modelo.feature_importances_), key=lambda p: -p[1]
):
    print(f'  {col:26s} {imp:.3f}')

print(
    '\n' + '=' * 68 + '\n'
    'RECORDATORIO: estas metricas NO validan nada sobre la Faja real.\n'
    'No existe un conjunto de prueba publico con resultados de perforacion.\n'
    'Lo que miden es si el modelo reprodujo la formula sintetica. Nada mas.\n'
    + '=' * 68
)

## 3. Exportar el grid a GeoJSON

Cada celda se convierte en un cuadrado con la probabilidad predicha en
`score`. El campo `demo: true` viaja con cada registro: el frontend lo usa
para no dibujar esta capa sin su banner.

In [ ]:
CELDA = 0.08  # mismo lado que el generador
FECHA = '2026-09-09'

datos['score'] = modelo.predict_proba(X)[:, 1]

features = []
for _, fila in datos.iterrows():
    lng, lat = float(fila['lng']), float(fila['lat'])
    features.append(
        {
            'type': 'Feature',
            'geometry': {
                'type': 'Polygon',
                'coordinates': [
                    [
                        [lng, lat],
                        [lng + CELDA, lat],
                        [lng + CELDA, lat + CELDA],
                        [lng, lat + CELDA],
                        [lng, lat],
                    ]
                ],
            },
            'properties': {
                'id': f"DEMO-{lng:.4f}-{lat:.4f}",
                'tipo': 'demo',
                'sector': 'demo',
                'estado': 'desconocido',
                'score': round(float(fila['score']), 4),
                'demo': True,
                'porosidad_pct': round(float(fila['porosidad_pct']), 1),
                'espesor_arena_neta_ft': round(
                    float(fila['espesor_arena_neta_ft']), 1
                ),
                'fuente': (
                    'Modelo DEMO sobre datos sinteticos. Distribuciones de la '
                    'tabla 1 del USGS FS 2009-3028; etiqueta y reparto '
                    'espacial inventados. NO es un dato observado.'
                ),
                'confianza': 'baja',
                'ultima_verificacion': FECHA,
            },
        }
    )

salida = {'type': 'FeatureCollection', 'features': features}
with open('prob_grid.geojson', 'w', encoding='utf-8') as f:
    json.dump(salida, f)

print(f'{len(features)} celdas escritas')
print(f"score: min {datos['score'].min():.3f} | max {datos['score'].max():.3f}")

In [ ]:
files.download('prob_grid.geojson')

## 4. Instalarlo en el mapa

1. Guarda `prob_grid.geojson` en `public/data/`.
2. `npm run data:validate`
3. `npm run dev` — aparece el interruptor de la capa DEMO.

La capa **no se dibuja sin su banner**. Esa condicion esta en el codigo, no en
la buena voluntad de quien la use: ver `dibujarGridProbabilidad` en
`src/map.js`.

### Antes de ensenarselo a nadie

Muestra la pantalla a alguien que no conozca el proyecto y preguntale:
*¿esto es un dato real?*

Si duda aunque sea un segundo, el etiquetado no basta. Refuerzalo.